# 25 · Batting jobs and playing styles: anchor, accelerator, hitter

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

Batters are grouped first by **job** (opener, top order, middle order, finisher, lower order, from how they are used) and then by **style** on normal balls, against an average batter in the same situation:

| | More boundaries than expected | Fewer boundaries than expected |
|---|---|---|
| **Fewer dots** | Accelerator | Anchor |
| **More dots** | Hitter | Slower scorer |

Then each batter is compared only with his own job, his own style, or both. Same 52 checks, rule fixed in advance.

**Result.** Comparing within style ranks the whole pool slightly better on random halves, but its top 10 adds fewer runs (9.6 against 11.6 per 100 pressure balls) and it is worse one season ahead. Overall: 0.137 (job), 0.126 (style), 0.124 (both) against 0.143 for the current rating, so the current rating stays. Style labels are also unstable: only 42% of players get the same label from two halves of their matches. The best pressure batter in each job and style is kept as a squad-planning view.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P28 - Batting roles AND playing styles (anchor / accelerator / hitter): should batters be compared within them? (research)  
  
User's idea (2026-11-15): bowlers are compared pace v pace and spin v spin; do the same for batters: first by job  
(opener, top order, middle order, finisher), then inside that by how they play (anchor, accelerator, hitter).  
Exploration before this script (no ranking result looked at): k-means on players' style numbers finds no natural groups  
(silhouette 0.24-0.27 for every k from 2 to 6), so styles are a sliding scale. A plain cricket rule is used instead.  
  
Definitions fixed BEFORE running:  
  Role  = the job, from P07's usage roles (batting position mix, entry over, phase shares; never performance).  
  Style = how he plays on NORMAL (non-pressure) balls, against an average batter in the same situation  
          (residuals from the average-player model), so the pressure balls being ranked do not define his group:  
            power    = boundaries per ball minus expected      rotation = dots per ball minus expected  
            Accelerator: power >= 0 and fewer dots than expected (scores both ways)  
            Hitter:      power >= 0 and more dots than expected  (boom or bust)  
            Anchor:      power <  0 and fewer dots than expected (keeps strike moving, fewer boundaries)  
            Slower scorer:   power <  0 and more dots than expected  
            Unclassified: under 120 normal balls faced  
          Thresholds are 0 (the average batter), not tuned.  
Versions tested (each ball's skill score minus the pressure-weighted average for the player's group, training matches only):  
  v11 role-relative, v12 style-relative, v13 role x style relative (cells with under 5 players fall back to the role).  
Rule: a version replaces the current rating only if its objective is within one paired standard error of the current  
one or better AND it changes the top 10. Otherwise styles are reported as a second view (best pressure batter in each  
role and style), not used for ranking. Sequential, 2 threads, low priority.
</details>

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from rrlib.common import CFG, SEED, load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman, excluded_reasons
from rrlib import ratings as RL
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p28_batting_styles")
pd.set_option("display.width", 220)
v = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
roles = load("roles")
ROLE = roles[roles.side == "bat"].set_index("player_id").role
v["w_"] = v.pressure_pct.clip(1e-3, 1) ** 2
v["role"] = v.striker_id.map(ROLE).fillna("Unassigned").values
MIN_NORMAL = 120


def styles_from(df):
    b = df[df.faced.astype(bool) & ~df.is_pressure.astype(bool)]
    g = b.groupby("striker_id").agg(n=("r_bdry_bat", "size"), power=("r_bdry_bat", "mean"),
                                    dots=("r_dot_bat", "mean"), outs=("r_wkt_bat", "mean"))
    st = np.select([(g.power >= 0) & (g.dots < 0), (g.power >= 0) & (g.dots >= 0), (g.power < 0) & (g.dots < 0)],
                   ["Accelerator", "Hitter", "Anchor"], "Slower scorer")
    g["style"] = np.where(g.n >= MIN_NORMAL, st, "Unclassified")
    return g


def add_relative(df, fit_on):
    sty = styles_from(fit_on)["style"]
    out = df.copy()
    out["style"] = out.striker_id.map(sty).fillna("Unclassified").values
    out["rs"] = out.role + " | " + out["style"]
    fo = fit_on.assign(style=fit_on.striker_id.map(sty).fillna("Unclassified").values)
    fo["rs"] = fo.role + " | " + fo["style"]
    f = fo[fo.faced.astype(bool) & fo.v3_bat.notna()].assign(wv=lambda d: d.w_ * d.v3_bat)

    def offs(col):
        o = f.groupby(col)[["wv", "w_"]].sum()
        return o.wv / o.w_
    o_role, o_sty, o_rs = offs("role"), offs("style"), offs("rs")
    n_rs = f.groupby("rs").striker_id.nunique()
    o_rs = o_rs[n_rs.reindex(o_rs.index) >= 5]                 # small cells fall back to the role
    out["v11_bat"] = out.v3_bat - out.role.map(o_role).fillna(0).values
    out["v12_bat"] = out.v3_bat - out["style"].map(o_sty).fillna(0).values
    rs_off = out.rs.map(o_rs)
    out["v13_bat"] = out.v3_bat - rs_off.fillna(out.role.map(o_role)).fillna(0).values
    return out


HOLD = CFG["cv"]["holdout_season"]
V = v[v.year < HOLD]
splits = [("rolling", f"{a[0]}-{a[-1]}->{b_}", V.year.isin(a).values, (V.year == b_).values) for a, b_ in CFG["cv"]["rolling"]]
mids = np.sort(V.match_id.unique()); rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False)); m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))
splits.append(("holdout", "2022-25->2026", (v.year < HOLD).values, (v.year == HOLD).values))
VERS = {"v3": "current", "v11": "same role", "v12": "same style", "v13": "same role and style"}
rows = []
with Timer(log, "52 checks + 2026, sequential"):
    for kind, name, tr_m, te_m in splits:
        base = v if kind == "holdout" else V
        tr = add_relative(base[tr_m], base[tr_m])
        te = base[te_m]
        t = te[te.is_pressure & te.v2_bat.notna()]
        yy = t.groupby("striker_id").agg(y2=("v2_bat", "mean"), y1=("v1_bat", "mean"), n=("v2_bat", "size")); yy = yy[yy.n >= 20]
        pool = yy.index[yy.index.isin(set(tr.loc[tr.is_pressure & tr.v1_bat.notna(), "striker_id"]))]
        if len(pool) < 15:
            continue
        for met in VERS:
            sc = RL.build_scores(tr, "bat", dict(chosen, value=met), comp).score.reindex(pool).fillna(-1e9)
            top = sc.sort_values(ascending=False).index[:10]
            wm = lambda col, idx: np.average(yy.loc[idx, col], weights=yy.loc[idx, "n"])
            rows.append({"kind": kind, "split": name, "version": met,
                         "rho": weighted_spearman(sc.values, yy.loc[pool].y2.values, yy.loc[pool].n.values),
                         "top10_lift_runs": 100 * (wm("y1", top) - wm("y1", pool))})
CV = pd.DataFrame(rows)
save_table(CV, "research_batting_styles_cv")
summ = {}
for met in VERS:
    r = CV[(CV.version == met) & (CV.kind != "holdout")]
    summ[met] = {"objective": float(0.5 * r[r.kind == "random"].rho.mean() + 0.5 * r[r.kind == "rolling"].rho.mean()),
                 "rolling": float(r[r.kind == "rolling"].rho.mean()), "top10_lift_runs": float(r.top10_lift_runs.mean()),
                 "holdout_2026": float(CV[(CV.version == met) & (CV.kind == "holdout")].rho.iloc[0])}
piv = CV[CV.kind == "random"].pivot(index="split", columns="version", values="rho")
for met in ("v11", "v12", "v13"):
    dd = piv[met] - piv["v3"]
    summ[met].update({"vs_current_diff": float(dd.mean()), "vs_current_se": float(dd.std(ddof=1) / np.sqrt(len(dd))),
                      "better_splits": int((dd > 0).sum())})

# ---------------------------------------------------------------- full data: lists, style mix, style map
full = add_relative(v, v)
STY = styles_from(v)
EX = set(excluded_reasons()); names = load("players").set_index("player_id").name
submitted = load_table("final_top30_bat").head(10).name.tolist()
FIN = load_table("final_all_bat").set_index("player_id")
pool_ids = FIN.index[~FIN.index.isin(EX)]
for met in VERS:
    sc = RL.build_scores(full, "bat", dict(chosen, value=met), comp)
    sc = sc[~sc.index.isin(EX)].sort_values("score", ascending=False)
    top = sc.index[:10]
    summ[met]["top10"] = [names.get(p) for p in top]
    summ[met]["same_names_as_submitted"] = len(set(summ[met]["top10"]) & set(submitted))
best = max(("v11", "v12", "v13"), key=lambda m_: summ[m_]["objective"])
ok = summ[best]["objective"] >= summ["v3"]["objective"] - summ[best]["vs_current_se"]
summ["decision"] = (f"{VERS[best]} is as good and changes the list: consider it" if ok and summ[best]["same_names_as_submitted"] < 10
                    else "keep the current rating; use roles and styles as a second view only")

M = FIN.loc[pool_ids, ["final_rank", "score", "p_top10", "n_p"]].copy()
M["name"] = names.reindex(M.index).values
M["role"] = ROLE.reindex(M.index).fillna("Unassigned").values
M = M.join(STY[["n", "power", "dots", "outs", "style"]])
M["style"] = M["style"].fillna("Unclassified")
M = M.sort_values("final_rank")
save_table(M.reset_index(), "batting_style_map")
mix = pd.DataFrame({"pool": M["style"].value_counts(), "top30": M.head(30)["style"].value_counts(),
                    "top10": M.head(10)["style"].value_counts()}).fillna(0).astype(int)
summ["style_mix"] = mix.to_dict()
# style labels repeatable? split matches in half, label each half separately (players with 60+ normal balls in both)
rng2 = np.random.RandomState(SEED + 1); agree = []
for _ in range(20):
    A = set(rng2.choice(v.match_id.unique(), v.match_id.nunique() // 2, replace=False))
    h1, h2 = styles_from(v[v.match_id.isin(A)]), styles_from(v[~v.match_id.isin(A)])
    ix = h1.index[h1.n >= 60].intersection(h2.index[h2.n >= 60])
    lab = lambda h: np.select([(h.power >= 0) & (h.dots < 0), (h.power >= 0) & (h.dots >= 0), (h.power < 0) & (h.dots < 0)],
                              ["Accelerator", "Hitter", "Anchor"], "Slower scorer")
    agree.append(float((lab(h1.loc[ix]) == lab(h2.loc[ix])).mean()))
summ["style_label_same_in_both_halves"] = float(np.mean(agree))
# best pressure batter in each role x style (current rating, available players only)
cells = (M[M["style"] != "Unclassified"].groupby(["role", "style"])
         .apply(lambda d: pd.Series({"players": len(d), "best": d["name"].iloc[0], "best_rank": int(d.final_rank.iloc[0]),
                                     "next": ", ".join(d["name"].iloc[1:3])})).reset_index())
save_table(cells, "batting_role_style_best")
save_json(summ, "research_batting_styles_summary")

log.info("\n" + pd.DataFrame({k: {kk: vv for kk, vv in d.items() if kk not in ("top10",)}
                              for k, d in summ.items() if isinstance(d, dict) and "objective" in d}).round(4).to_string())
for met, lab_ in VERS.items():
    log.info(f"{lab_}: {summ[met]['top10']}")
log.info("style mix (players):\n" + mix.to_string())
log.info(f"style label same in both halves of the matches: {summ['style_label_same_in_both_halves']:.2f}")
log.info("style averages (pool):\n" + M.groupby("style")[["power", "dots", "outs"]].mean().mul(100).round(1).to_string())
log.info("top 10 with style:\n" + M.head(10)[["name", "role", "style", "power", "dots", "outs"]].round(3).to_string())
log.info("best in each role and style:\n" + cells.to_string(index=False))
log.info(f"decision: {summ['decision']}")
